# Evaluation Agent input timeline — file-based parity test

这个 notebook 不调用 Evaluation Agent 或写入 MySQL。它的目标是验证 **从三类原始资料到 Evaluation Agent input** 的数据处理。

- `chat history`：JSON
- `speech transcript`：JSON
- `interaction log`：CSV

文件的字段名可以与平台数据库不同；第 2 个 cell 是唯一的适配层。适配完成后，notebook 会编译并直接调用平台的 `buildTimelineEventsFromRows`。该函数正是线上 `eventTimelineBuilder.ts` 在从三个 source cursor 取到增量资料后使用的同一个预处理核心，而不是在 notebook 中复制一份规则。

输出 JSON 中的 `evaluationAgentInput` 的结构与 audit 中存储的 `evaluation_input_payload` 一致：`{ initiation, partition }`。

In [ ]:
from pathlib import Path

PROJECT = Path('/Users/yanliu/Projects/Manipulation research-platform')
BACKEND = PROJECT / 'backend'

# Pilot_user04 raw exports. Keep research data outside the repository.
RAW_DATA_DIR = Path('/Users/yanliu/uploads_20261015_143157')
CHAT_HISTORY_FILE = RAW_DATA_DIR / 'chat_history' / 'chat_history_Pilot_user04_1792044796118.json'
SPEECH_TRANSCRIPT_FILE = RAW_DATA_DIR / 'transcripts' / 'transcript_Pilot_user04_1792044796118.json'
INTERACTION_LOG_FILE = RAW_DATA_DIR / 'interaction_logs' / 'interaction_logs_Pilot_user04_1792044796118.csv'

# The platform stores interaction_logs.timestamp as milliseconds relative to the task
# start. Set this to the session's client sessionStartEpochMs. For historical files with
# absolute interaction timestamps, the adapter below converts them to this same clock.
SESSION_START_EPOCH_MS = 1792044796118  # Pilot_user04 session-id suffix / session-start epoch

# Current platform speech rows are absolute epoch milliseconds. This anchor matters only
# when testing older speech files whose timestamps are relative to recording start.
# Set this to the exact offset used in your offline preprocessing pipeline.
# adjusted_speech_end = raw_timestamp_end + TRANSCRIPT_TIME_OFFSET_MS
# Pilot_user04's transcript clock begins 2,686 ms after the session clock. This is
# verified against its audit: transcript timestamp 71 maps to epoch session + 2,757 ms.
TRANSCRIPT_TIME_OFFSET_MS = 2686
RECORDING_START_EPOCH_MS = SESSION_START_EPOCH_MS + TRANSCRIPT_TIME_OFFSET_MS

# These three Pilot_user04 exports use milliseconds relative to the session / recording start.
# For a current platform database export, set CHAT_TIMESTAMPS_ARE_RELATIVE_MS = False;
# current speech_segments are already absolute and are recognised by the shared backend code.
CHAT_TIMESTAMPS_ARE_RELATIVE_MS = True

# These are the three durable timeline_source_cursors *before* this materialisation.
# Keep 0 for a complete first pass. To reproduce a later incremental trigger, set each
# value to the last source id that was already consumed by an earlier pass.
SOURCE_CURSORS_BEFORE = {
    'interaction_logs': 0,
    'speech_segments': 0,
    'messages': 0,
}

# At a live initiation, only rows already written to the database are visible. Leave
# these as None for the complete-data comparison. For an incremental replay, set the
# high-watermark id for each source at this particular initiation (see the printed
# suggested checkpoints below).
SOURCE_MAX_AVAILABLE = {
    'interaction_logs': None,
    'speech_segments': None,
    'messages': None,
}

# Primary historical-replay switch: treat this as the moment Evaluation Agent was
# initiated, measured in milliseconds since session start. Only source rows whose
# participant-time is at or before this point are visible. Speech uses its adjusted
# end time (raw timestamp_end + TRANSCRIPT_TIME_OFFSET_MS). Keep None for full data.
# Run several fixed timepoints sequentially to test cursor behaviour; include one
# final point at/after the latest source timestamp to prove complete coverage.
INITIATED_AT_RELATIVE_MS = None

# Mirrors manipulation_state.last_evaluated_event_id. Existing materialised event ids can
# start above 1; choose the next id and evaluator cursor to reproduce that state.
NEXT_EVENT_ID = 1
LAST_EVALUATED_EVENT_ID = 0

INITIATION = {
    'triggerType': 'automatic_analysis',
    'reason': 'Notebook reconstruction from three source files.',
}

# Includes both the complete materialized unified timeline and the derived agent input.
OUTPUT_FILE = RAW_DATA_DIR / 'unified_timeline_Pilot_user04_1792044796118.json'
PARTITION_PREVIEW_EVENTS = 20  # Set None to print every event in the partition.
PRINT_FULL_EVALUATION_INPUT = False  # The full object is always written to OUTPUT_FILE.

if not SESSION_START_EPOCH_MS:
    raise ValueError('Set SESSION_START_EPOCH_MS before running the notebook.')
for source_file in (CHAT_HISTORY_FILE, SPEECH_TRANSCRIPT_FILE, INTERACTION_LOG_FILE):
    if not source_file.exists():
        raise FileNotFoundError(f'Missing test input: {source_file}')


In [ ]:
import csv
import json
from datetime import datetime, timezone

def unwrap_rows(value, preferred_keys):
    """Accept a list or common JSON wrappers without changing platform semantics."""
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in preferred_keys:
            if isinstance(value.get(key), list):
                return value[key]
    raise ValueError(f'Expected an array or one of {preferred_keys}; got {type(value).__name__}')

def read_json_rows(path, preferred_keys):
    return unwrap_rows(json.loads(path.read_text(encoding='utf-8')), preferred_keys)

def pick(row, *names, default=None):
    # Some CSV exports use different capitalization.
    normalized = {str(k).lower(): v for k, v in row.items()}
    for name in names:
        if name in row and row[name] not in (None, ''):
            return row[name]
        value = normalized.get(name.lower())
        if value not in (None, ''):
            return value
    return default

def as_int(value, fallback=0):
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return fallback

def as_json_or_value(value):
    if not isinstance(value, str):
        return value
    stripped = value.strip()
    if not stripped:
        return None
    try:
        return json.loads(stripped)
    except json.JSONDecodeError:
        return value

def as_bool(value):
    if isinstance(value, str):
        return value.strip().lower() in ('1', 'true', 'yes')
    return bool(value)

def to_iso_timestamp(value, relative_ms=False):
    """Return a value JavaScript new Date(value) will parse identically."""
    if isinstance(value, (int, float)) or (isinstance(value, str) and value.strip().replace('.', '', 1).isdigit()):
        epoch = float(value)
        if relative_ms:
            epoch = SESSION_START_EPOCH_MS + epoch
        elif epoch < 100_000_000_000:  # seconds rather than milliseconds
            epoch *= 1000
        return datetime.fromtimestamp(epoch / 1000, tz=timezone.utc).isoformat().replace('+00:00', 'Z')
    return str(value)

def to_relative_session_ms(value, origin_epoch_ms=None):
    """Timestamp used only to simulate which rows existed at an initiation time."""
    numeric = as_int(value)
    if numeric >= 100_000_000_000:  # absolute epoch milliseconds
        return numeric - SESSION_START_EPOCH_MS
    return (origin_epoch_ms or SESSION_START_EPOCH_MS) + numeric - SESSION_START_EPOCH_MS

# ---- File-format adapter only: map historical exports to the platform raw-row schema. ----
# Once these objects are made, no notebook-specific event-processing logic is used.
chat_rows = read_json_rows(CHAT_HISTORY_FILE, ['messages', 'chat_history', 'data', 'items'])
speech_rows = read_json_rows(SPEECH_TRANSCRIPT_FILE, ['segments', 'transcript', 'speech_segments', 'data', 'items'])
with INTERACTION_LOG_FILE.open(encoding='utf-8-sig', newline='') as f:
    interaction_rows = list(csv.DictReader(f))

interaction_logs = []
for index, row in enumerate(interaction_rows, start=1):
    raw_timestamp = as_int(pick(row, 'timestamp', 'timestamp_ms', 'relative_timestamp_ms', default=0))
    # Production expects a relative interaction-log timestamp. This makes absolute CSV
    # exports equivalent without altering the production preprocessing function.
    relative_timestamp = raw_timestamp - SESSION_START_EPOCH_MS if raw_timestamp >= 100_000_000_000 else raw_timestamp
    interaction_logs.append({
        'id': as_int(pick(row, 'id', 'log_id', 'event_id'), index),
        'event_type': str(pick(row, 'event_type', 'type', default='ui')),
        'action': str(pick(row, 'action', default='unknown')),
        'target_element': pick(row, 'target_element', 'element'),
        'target_section': pick(row, 'target_section', 'section'),
        'text_content': pick(row, 'text_content', 'text'),
        'metadata': as_json_or_value(pick(row, 'metadata')),
        'timestamp': relative_timestamp,
        '_source_time_ms': relative_timestamp,
    })

speech_segments = []
for index, row in enumerate(speech_rows, start=1):
    speech_segments.append({
        'id': as_int(pick(row, 'id', 'segment_id', 'speech_id'), index),
        # Raw transcript exports use timestamp / receiveTimestamp; DB exports use
        # timestamp_start / timestamp_end. Both map to the same backend schema.
        'timestamp_start': as_int(pick(row, 'timestamp_start', 'start_time', 'start_ms', 'start', 'timestamp', default=0)),
        'timestamp_end': as_int(pick(row, 'timestamp_end', 'end_time', 'end_ms', 'end', 'receiveTimestamp', default=0)),
        'transcription': str(pick(row, 'transcription', 'text', 'content', default='')),
        # adjusted speech end on the session clock: raw_end + transcript offset
        '_source_time_ms': to_relative_session_ms(pick(row, 'timestamp_end', 'end_time', 'end_ms', 'end', 'receiveTimestamp', default=0), RECORDING_START_EPOCH_MS),
    })

messages = []
for index, row in enumerate(chat_rows, start=1):
    role = str(pick(row, 'role', 'sender', default='user')).lower()
    if role in ('ai', 'bot'): role = 'assistant'
    # Direct guidance is preserved as its own chat-history role; the shared backend builder emits it as an ai_response.
    if role not in ('user', 'assistant', 'ai_guidance'):
        raise ValueError(f'Unsupported chat role {role!r} in row {index}')
    raw_timestamp = pick(row, 'timestamp', 'created_at', 'time')
    messages.append({
        'id': as_int(pick(row, 'id', 'message_id'), index),
        'role': role,
        'content': str(pick(row, 'content', 'message', 'text', default='')),
        'is_guidance': as_bool(pick(row, 'is_guidance', 'guidance', default=False)),
        'timestamp': to_iso_timestamp(raw_timestamp, CHAT_TIMESTAMPS_ARE_RELATIVE_MS),
        '_source_time_ms': to_relative_session_ms(raw_timestamp, SESSION_START_EPOCH_MS) if CHAT_TIMESTAMPS_ARE_RELATIVE_MS else to_relative_session_ms(raw_timestamp),
    })

print({'rawRows': {'interaction_logs': len(interaction_logs), 'speech_segments': len(speech_segments), 'messages': len(messages)}})


In [ ]:
# This is the same `WHERE id > last_source_id` selection used by
# eventTimelineBuilder.materializeTimelineDeltaOnce. The time condition is a historical
# replay approximation; the optional id upper bound models a known DB snapshot exactly.
source_rows = {
    'interaction_logs': interaction_logs,
    'speech_segments': speech_segments,
    'messages': messages,
}
delta = {
    source: [
        row for row in sorted(rows, key=lambda item: item['id'])
        if row['id'] > SOURCE_CURSORS_BEFORE[source]
        and (INITIATED_AT_RELATIVE_MS is None or row['_source_time_ms'] <= INITIATED_AT_RELATIVE_MS)
        and (SOURCE_MAX_AVAILABLE[source] is None or row['id'] <= SOURCE_MAX_AVAILABLE[source])
    ]
    for source, rows in source_rows.items()
}
source_cursors_after = {
    source: (rows[-1]['id'] if rows else SOURCE_CURSORS_BEFORE[source])
    for source, rows in delta.items()
}

# The production materializer reads this one already-consumed row only as
# placement context for an assistant-only message delta; it is not re-emitted.
prior_user_messages = [
    row for row in messages
    if row['role'] == 'user' and row['id'] <= SOURCE_CURSORS_BEFORE['messages']
]
previous_user_message_timestamp = (
    max(prior_user_messages, key=lambda row: row['id'])['timestamp']
    if prior_user_messages else None
)

print('cursor before:', SOURCE_CURSORS_BEFORE)
print('initiation time (relative ms):', INITIATED_AT_RELATIVE_MS)
print('new raw rows:', {source: len(rows) for source, rows in delta.items()})
print('cursor after: ', source_cursors_after)

# Mechanical checkpoints validate the no-duplicate rule, but they approximate historical
# initiations. To reproduce an actual initiation, use the source max ids present in the
# corresponding evaluation audit or database snapshot.
for source, rows in source_rows.items():
    ordered_ids = [row['id'] for row in sorted(rows, key=lambda item: item['id'])]
    if ordered_ids:
        suggested = [ordered_ids[round((len(ordered_ids) - 1) * fraction)] for fraction in (0.25, 0.50, 0.75, 1.0)]
        print(f'{source} id range: {ordered_ids[0]}–{ordered_ids[-1]}; suggested 25/50/75/100% maxima: {suggested}')

# Reproducible random initiation times for a cursor replay. These are source-time
# checkpoints, not claimed historical database-arrival times.
import random
available_times = sorted({row['_source_time_ms'] for rows in source_rows.values() for row in rows})
rng = random.Random(20261013)
interior_times = [time for time in available_times if time not in (available_times[0], available_times[-1])]
random_points = sorted(rng.sample(interior_times, min(3, len(interior_times)))) if interior_times else []
print('Suggested reproducible initiation times (ms since session start):', random_points + ([available_times[-1]] if available_times else []))


In [ ]:
import subprocess
import tempfile

# Compile, then invoke the actual backend preprocessing export. This guarantees that a
# future edit to eventTimelineBuilder.ts is picked up by this parity test.
subprocess.run(['npm', 'run', 'build'], cwd=BACKEND, check=True)

payload = {
    'sessionStartEpochMs': SESSION_START_EPOCH_MS,
    'recordingStartEpochMs': RECORDING_START_EPOCH_MS,
    'interactionLogs': delta['interaction_logs'],
    'speechSegments': delta['speech_segments'],
    'messages': delta['messages'],
    'previousUserMessageTimestamp': previous_user_message_timestamp,
    'nextEventId': NEXT_EVENT_ID,
}
node_script = r'''
const fs = require('fs');
const { buildTimelineEventsFromRows, unevaluatedTail } = require('./dist/services/eventTimelineBuilder');
const input = JSON.parse(fs.readFileSync(process.argv[1], 'utf8'));
const events = buildTimelineEventsFromRows(input.payload);
const partition = unevaluatedTail(events, input.lastEvaluatedEventId);
process.stdout.write(JSON.stringify({ events, partition }));
'''

with tempfile.TemporaryDirectory() as temp_dir:
    input_path = Path(temp_dir) / 'timeline_input.json'
    input_path.write_text(json.dumps({'payload': payload, 'lastEvaluatedEventId': LAST_EVALUATED_EVENT_ID}, ensure_ascii=False), encoding='utf-8')
    completed = subprocess.run(
        ['node', '-e', node_script, str(input_path)],
        cwd=BACKEND, check=True, capture_output=True, text=True,
    )
result = json.loads(completed.stdout)
events = result['events']
partition = result['partition']

evaluation_agent_input = {
    'initiation': INITIATION,
    'partition': partition,
}

print(f'Materialized events: {len(events)} | unevaluated Evaluation-Agent partition: {len(partition)}')
events_to_print = partition if PARTITION_PREVIEW_EVENTS is None else partition[:PARTITION_PREVIEW_EVENTS]
for event in events_to_print:
    content = event['speech'] or event['userMessage'] or event['aiResponse'] or event['actionDescription']
    print(f"#{event['eventId']} {event['eventType']} [{event['timestampStart']}–{event['timestampEnd']}] {content}")
if len(events_to_print) < len(partition):
    print(f'… {len(partition) - len(events_to_print)} more events are in {OUTPUT_FILE.name}.')


In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
audit_preview = {
    'initiatedAtRelativeMs': INITIATED_AT_RELATIVE_MS,
    'sourceCursorsBefore': SOURCE_CURSORS_BEFORE,
    'sourceCursorsAfter': source_cursors_after,
    'rawDeltaCounts': {source: len(rows) for source, rows in delta.items()},
    'lastEvaluatedEventIdBefore': LAST_EVALUATED_EVENT_ID,
    'materializedEvents': events,
    'evaluationAgentInput': evaluation_agent_input,
}
OUTPUT_FILE.write_text(json.dumps(audit_preview, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Wrote: {OUTPUT_FILE}')

# This is exactly the payload portion retained by evaluation_agent_audit.
if PRINT_FULL_EVALUATION_INPUT:
    print(json.dumps(evaluation_agent_input, ensure_ascii=False, indent=2))
else:
    print('Set PRINT_FULL_EVALUATION_INPUT = True to print the complete payload in the notebook.')
